# num → en | Bahdanau Attention

담당: 김태우

기존 `Decoder`에 `attention='bahdanau'`를 추가하고 Luong/Bahdanau/No Attention을 비교했습니다. 추가 요청에 따라 김서현 님의 `oversample_short()` 방식을 활용해 1~2자리 숫자만 Train에서 오버샘플링한 Bahdanau 모델을 학습합니다.

원래 세 모델의 출력은 이전 Colab 실험 결과입니다. 새 오버샘플링 실험 셀은 실행 결과를 직접 생성해야 하며, 재학습 결과가 기존 성능과 다를 수 있습니다.


## 실행 환경 및 설정

In [1]:
%cd /content
!test -d /content/2026-nlp/.git || git clone https://github.com/jungyy808/2026-nlp.git
%cd /content/2026-nlp/project01/work


/content
Cloning into '2026-nlp'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 87 (delta 13), reused 27 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 35.56 MiB | 11.76 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/2026-nlp/project01/work


In [2]:
import os
import random
import time
import zlib
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SRC, TGT = 'num', 'en'
DATA_DIR = '../data'
SAVE_DIR = '..'
CONFIG = dict(
    emb_dim=128,
    hid_dim=256,
    n_layers=2,
    dropout=0.2,
    rnn='gru',
    attention='luong'
)
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
SEED = 42
MAX_TRAIN = None

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


device: cuda


## 데이터 및 Vocabulary

In [3]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    src_vocab = Vocab(src_lang, [s for s, _, _ in pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)


In [4]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음 (Colab이면 clone 셀 확인)'
print('pt 저장 위치:', os.path.abspath(SAVE_DIR))

pairs = get_pairs(SRC, TGT)
train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN)

MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs) + 10
print(f'{SRC} -> {TGT}: 전체 {len(pairs):,}쌍 | train {len(train_loader.dataset):,} / '
      f'valid {len(valid_loader.dataset):,} / test {len(test_loader.dataset):,}')
print(f'vocab: src {len(src_vocab)}, tgt {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')
for s, t, _ in pairs[:3]:
    print(f'  {s!r} -> {t!r}')


작업 위치: /content/2026-nlp/project01/work
pt 저장 위치: /content/2026-nlp/project01
num -> en: 전체 100,000쌍 | train 80,028 / valid 9,967 / test 10,005
vocab: src 14, tgt 36 | 디코딩 최대 길이 26
  '3622185' -> 'three million six hundred twenty two thousand one hundred eighty five'
  '4507' -> 'four thousand five hundred seven'
  '49330605' -> 'forty nine million three hundred thirty thousand six hundred five'


## 모델

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden

### Bahdanau Attention 구현

기존 Luong/None 분기를 유지하면서 additive score를 계산하는 Bahdanau 분기를 추가합니다. 이전 디코더 hidden state로 attention context를 계산한 뒤 GRU 입력과 결합합니다.

In [6]:
# 기존 Decoder에 Bahdanau 옵션을 추가해 별도 클래스를 만들지 않고 공통 인터페이스를 유지한다.
# 이전 Decoder hidden state로 입력 위치를 정하고 context를 GRU 입력에 결합한다.
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim,
             n_layers, dropout, rnn, attention):
        super().__init__()

        assert attention in ('luong', 'bahdanau', None)
        assert rnn in ('gru', 'lstm')

        self.attention = attention
        self.emb = nn.Embedding(
            vocab_size, emb_dim, padding_idx=PAD
        )

        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU

        rnn_input_dim = (
            emb_dim + hid_dim
            if attention == 'bahdanau'
            else emb_dim
        )

        self.rnn = RNN(
            rnn_input_dim,
            hid_dim,
            n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )

        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)

        elif attention == 'bahdanau':
            self.W_h = nn.Linear(hid_dim, hid_dim, bias=False)
            self.W_s = nn.Linear(hid_dim, hid_dim, bias=False)
            self.v_a = nn.Linear(hid_dim, 1, bias=False)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)

        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)

        if self.attention == 'bahdanau':
            query = hidden[0][-1] if isinstance(hidden, tuple) else hidden[-1]

            scores = self.v_a(
                torch.tanh(
                    self.W_h(enc_out) +
                    self.W_s(query).unsqueeze(1)
                )
            ).squeeze(-1)

            scores = scores.masked_fill(~src_mask, -1e9)
            weights = F.softmax(scores, dim=1)

            ctx = torch.bmm(
                weights.unsqueeze(1), enc_out
            ).squeeze(1)

            rnn_input = torch.cat(
                [x, ctx.unsqueeze(1)], dim=2
            )
            h, hidden = self.rnn(rnn_input, hidden)
            h = h.squeeze(1)

            h = torch.tanh(
                self.combine(torch.cat([h, ctx], dim=1))
            )

        else:
            h, hidden = self.rnn(x, hidden)
            h = h.squeeze(1)

            if self.attention == 'luong':
                scores = torch.bmm(
                    enc_out, self.attn(h).unsqueeze(2)
                ).squeeze(2)
                scores = scores.masked_fill(~src_mask, -1e9)

                ctx = torch.bmm(
                    F.softmax(scores, dim=1).unsqueeze(1),
                    enc_out
                ).squeeze(1)

                h = torch.tanh(
                    self.combine(torch.cat([h, ctx], dim=1))
                )

        return self.out(self.drop(h)), hidden

In [7]:
class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong'):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)

## 학습 및 디코딩 함수

In [8]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def beam_decode(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams = [([], 0.0)]
    finished = []
    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / len(seq)))
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams or len(finished) >= beam:
            break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / len(seq)) for seq, score in beams]
    best = max(finished, key=lambda f: f[1])[0]
    return tgt_vocab.decode(best)


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR, save_path=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history


In [9]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')


In [14]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt


## Luong Attention

In [ ]:
set_seed(SEED)

luong_config = {**CONFIG, 'attention': 'luong'}

luong_model = Seq2Seq(
    len(src_vocab),
    len(tgt_vocab),
    **luong_config
).to(DEVICE)

print("Device:", DEVICE)
print("Attention:", luong_model.decoder.attention)
print("Train samples:", len(train_loader.dataset))
print("Valid samples:", len(valid_loader.dataset))
print("Parameters:", sum(p.numel() for p in luong_model.parameters()))

luong_history = train(
    luong_model,
    train_loader,
    valid_loader,
    epochs=EPOCHS,
    lr=LR
)


Device: cuda
Attention: luong
Train samples: 80028
Valid samples: 9967
Parameters: 1594916
epoch  1 | train loss 0.5292 | valid loss 0.0298 | valid acc 93.49% | 51s
epoch  2 | train loss 0.0324 | valid loss 0.0100 | valid acc 97.80% | 50s
epoch  3 | train loss 0.0164 | valid loss 0.0131 | valid acc 97.32% | 49s
epoch  4 | train loss 0.0149 | valid loss 0.0107 | valid acc 97.17% | 49s
epoch  5 | train loss 0.0148 | valid loss 0.0133 | valid acc 97.52% | 49s
epoch  6 | train loss 0.0113 | valid loss 0.0050 | valid acc 98.27% | 50s
epoch  7 | train loss 0.0086 | valid loss 0.0045 | valid acc 98.50% | 49s
epoch  8 | train loss 0.0086 | valid loss 0.0041 | valid acc 98.78% | 49s
epoch  9 | train loss 0.0109 | valid loss 0.0035 | valid acc 99.10% | 49s
epoch 10 | train loss 0.0072 | valid loss 0.0025 | valid acc 99.35% | 49s
epoch 11 | train loss 0.0083 | valid loss 0.0130 | valid acc 98.30% | 49s
epoch 12 | train loss 0.0077 | valid loss 0.0018 | valid acc 99.63% | 49s
epoch 13 | train loss

In [ ]:
luong_test_rows = predict_all(
    luong_model, test_loader, tgt_vocab
)

luong_test_acc = exact_match(luong_test_rows)

print(f"Luong Test Accuracy: {luong_test_acc * 100:.2f}%")
print(f"Test samples: {len(luong_test_rows):,}")

print("\n[자릿수별 정확도]")
acc_by_length(luong_test_rows)

print("\n[예측 사례]")
show_examples(luong_test_rows, n=5)


Luong Test Accuracy: 99.85%
Test samples: 10,005

[자릿수별 정확도]
자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 |  99.87%
     8 |   1472 |  99.93%
     9 |   1553 |  99.55%
    10 |   1448 |  99.65%


[예측 사례]
--- 맞은 예시 ---
입력: 897890
정답: eight hundred ninety seven thousand eight hundred ninety
예측: eight hundred ninety seven thousand eight hundred ninety

입력: 93532
정답: ninety three thousand five hundred thirty two
예측: ninety three thousand five hundred thirty two

입력: 3894895050
정답: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
예측: three billion eight hundred ninety four million eight hundred ninety five thousand fifty

입력: 9471425031
정답: nine billion four hundred seventy one million four hundred twenty five thousand thirty one
예측: nine billion four hundred seventy one million four hundred twenty five thousand

## Bahdanau Attention

In [ ]:
set_seed(SEED)

bahdanau_config = {
    **CONFIG,
    'attention': 'bahdanau'
}

bahdanau_model = Seq2Seq(
    len(src_vocab),
    len(tgt_vocab),
    **bahdanau_config
).to(DEVICE)

print("Device:", DEVICE)
print("Attention:", bahdanau_model.decoder.attention)
print("GRU input size:", bahdanau_model.decoder.rnn.input_size)
print("Train samples:", len(train_loader.dataset))
print("Valid samples:", len(valid_loader.dataset))
print("Parameters:", sum(
    p.numel() for p in bahdanau_model.parameters()
))

bahdanau_history = train(
    bahdanau_model,
    train_loader,
    valid_loader,
    epochs=EPOCHS,
    lr=LR
)


Device: cuda
Attention: bahdanau
GRU input size: 384
Train samples: 80028
Valid samples: 9967
Parameters: 1857316
epoch  1 | train loss 0.5140 | valid loss 0.0086 | valid acc 98.98% | 54s
epoch  2 | train loss 0.0168 | valid loss 0.0022 | valid acc 99.68% | 53s
epoch  3 | train loss 0.0051 | valid loss 0.0004 | valid acc 99.95% | 53s
epoch  4 | train loss 0.0041 | valid loss 0.0026 | valid acc 99.46% | 52s
epoch  5 | train loss 0.0015 | valid loss 0.0001 | valid acc 100.00% | 52s
epoch  6 | train loss 0.0006 | valid loss 0.0001 | valid acc 100.00% | 52s
epoch  7 | train loss 0.0046 | valid loss 0.0001 | valid acc 99.98% | 52s
epoch  8 | train loss 0.0013 | valid loss 0.0006 | valid acc 99.86% | 52s
epoch  9 | train loss 0.0007 | valid loss 0.0004 | valid acc 99.93% | 52s
epoch 10 | train loss 0.0031 | valid loss 0.0000 | valid acc 100.00% | 52s
epoch 11 | train loss 0.0017 | valid loss 0.0116 | valid acc 97.10% | 52s
epoch 12 | train loss 0.0028 | valid loss 0.0023 | valid acc 99.60% |

In [ ]:
bahdanau_test_rows = predict_all(
    bahdanau_model, test_loader, tgt_vocab
)

bahdanau_test_acc = exact_match(bahdanau_test_rows)

print(f"Bahdanau Test Accuracy: {bahdanau_test_acc * 100:.2f}%")
print(f"Test samples: {len(bahdanau_test_rows):,}")

print("\n[자릿수별 정확도]")
acc_by_length(bahdanau_test_rows)

print("\n[예측 사례]")
show_examples(bahdanau_test_rows, n=5)


Bahdanau Test Accuracy: 99.98%
Test samples: 10,005

[자릿수별 정확도]
자릿수 | 개수 | 정확도
     1 |      1 |   0.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 | 100.00%
     8 |   1472 | 100.00%
     9 |   1553 |  99.94%
    10 |   1448 | 100.00%

[예측 사례]
--- 맞은 예시 ---
입력: 897890
정답: eight hundred ninety seven thousand eight hundred ninety
예측: eight hundred ninety seven thousand eight hundred ninety

입력: 93532
정답: ninety three thousand five hundred thirty two
예측: ninety three thousand five hundred thirty two

입력: 3894895050
정답: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
예측: three billion eight hundred ninety four million eight hundred ninety five thousand fifty

입력: 9471425031
정답: nine billion four hundred seventy one million four hundred twenty five thousand thirty one
예측: nine billion four hundred seventy one million four hundred twenty five thousa

## No Attention

In [ ]:
set_seed(SEED)

no_att_config = {
    **CONFIG,
    'attention': None
}

no_att_model = Seq2Seq(
    len(src_vocab),
    len(tgt_vocab),
    **no_att_config
).to(DEVICE)

print("Attention:", no_att_model.decoder.attention)
print("GRU input size:", no_att_model.decoder.rnn.input_size)
print("Parameters:", sum(
    p.numel() for p in no_att_model.parameters()
))

no_att_history = train(
    no_att_model,
    train_loader,
    valid_loader,
    epochs=EPOCHS,
    lr=LR
)


Attention: None
GRU input size: 128
Parameters: 1398052
epoch  1 | train loss 0.9431 | valid loss 0.1166 | valid acc 73.63% | 32s
epoch  2 | train loss 0.1154 | valid loss 0.0194 | valid acc 96.64% | 31s
epoch  3 | train loss 0.0382 | valid loss 0.0070 | valid acc 98.99% | 30s
epoch  4 | train loss 0.0186 | valid loss 0.0051 | valid acc 98.88% | 30s
epoch  5 | train loss 0.0149 | valid loss 0.0030 | valid acc 99.46% | 31s
epoch  6 | train loss 0.0094 | valid loss 0.0045 | valid acc 98.75% | 30s
epoch  7 | train loss 0.0090 | valid loss 0.0029 | valid acc 99.17% | 30s
epoch  8 | train loss 0.0056 | valid loss 0.0022 | valid acc 99.37% | 29s
epoch  9 | train loss 0.0108 | valid loss 0.0025 | valid acc 99.38% | 30s
epoch 10 | train loss 0.0047 | valid loss 0.0008 | valid acc 99.77% | 29s
epoch 11 | train loss 0.0034 | valid loss 0.0006 | valid acc 99.84% | 29s
epoch 12 | train loss 0.0054 | valid loss 0.0010 | valid acc 99.73% | 29s
epoch 13 | train loss 0.0040 | valid loss 0.0014 | valid

In [ ]:
no_att_test_rows = predict_all(
    no_att_model, test_loader, tgt_vocab
)

no_att_test_acc = exact_match(no_att_test_rows)

print(f"No Attention Test Accuracy: {no_att_test_acc * 100:.2f}%")

print("\n[자릿수별 정확도]")
acc_by_length(no_att_test_rows)

print("\n[예측 사례]")
show_examples(no_att_test_rows, n=5)


No Attention Test Accuracy: 99.85%

[자릿수별 정확도]
자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 |  99.93%
     8 |   1472 |  99.93%
     9 |   1553 |  99.61%
    10 |   1448 |  99.52%

[예측 사례]
--- 맞은 예시 ---
입력: 897890
정답: eight hundred ninety seven thousand eight hundred ninety
예측: eight hundred ninety seven thousand eight hundred ninety

입력: 93532
정답: ninety three thousand five hundred thirty two
예측: ninety three thousand five hundred thirty two

입력: 3894895050
정답: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
예측: three billion eight hundred ninety four million eight hundred ninety five thousand fifty

입력: 9471425031
정답: nine billion four hundred seventy one million four hundred twenty five thousand thirty one
예측: nine billion four hundred seventy one million four hundred twenty five thousand thirty one

입력

## 실험 결과 비교

In [ ]:
experiments = {
    "Luong": (
        luong_model, luong_history, luong_test_acc
    ),
    "Bahdanau": (
        bahdanau_model, bahdanau_history, bahdanau_test_acc
    ),
    "No Attention": (
        no_att_model, no_att_history, no_att_test_acc
    )
}

print("=" * 75)
print(f"{'Model':<16} {'Best Epoch':>10} {'Valid Acc':>12} {'Test Acc':>12} {'Params':>12}")
print("=" * 75)

for name, (m, history, test_acc) in experiments.items():
    best = max(
        history,
        key=lambda x: x['valid_exact_match']
    )

    params = sum(p.numel() for p in m.parameters())

    print(
        f"{name:<16} "
        f"{best['epoch']:>10} "
        f"{best['valid_exact_match']*100:>11.2f}% "
        f"{test_acc*100:>11.2f}% "
        f"{params:>12,}"
    )


Model            Best Epoch    Valid Acc     Test Acc       Params
Luong                    15       99.65%       99.85%    1,594,916
Bahdanau                  5      100.00%       99.98%    1,857,316
No Attention             15       99.88%       99.85%    1,398,052


In [ ]:
luong_map = {
    key: (gold, pred)
    for _, gold, pred, key in luong_test_rows
}

bahdanau_map = {
    key: (gold, pred)
    for _, gold, pred, key in bahdanau_test_rows
}

improved = []
regressed = []

for key, (gold, bah_pred) in bahdanau_map.items():
    _, luong_pred = luong_map[key]

    if luong_pred != gold and bah_pred == gold:
        improved.append((key, gold, luong_pred, bah_pred))

    elif luong_pred == gold and bah_pred != gold:
        regressed.append((key, gold, luong_pred, bah_pred))

print("Luong 오답 → Bahdanau 정답:", len(improved))
print("Luong 정답 → Bahdanau 오답:", len(regressed))

print("\n[Bahdanau가 개선한 사례]")
for key, gold, luong_pred, bah_pred in improved[:5]:
    print(f"\n입력: {key}")
    print(f"정답: {gold}")
    print(f"Luong: {luong_pred}")
    print(f"Bahdanau: {bah_pred}")


Luong 오답 → Bahdanau 정답: 14
Luong 정답 → Bahdanau 오답: 1

[Bahdanau가 개선한 사례]

입력: 8567012011
정답: eight billion five hundred sixty seven million twelve thousand eleven
Luong: eight billion five hundred sixty seven million eleven thousand one hundred one
Bahdanau: eight billion five hundred sixty seven million twelve thousand eleven

입력: 9055018454
정답: nine billion fifty five million eighteen thousand four hundred fifty four
Luong: nine billion fifty five million seventeen thousand four hundred fifty four
Bahdanau: nine billion fifty five million eighteen thousand four hundred fifty four

입력: 777016393
정답: seven hundred seventy seven million sixteen thousand three hundred ninety three
Luong: seven hundred seventy seven million fourteen thousand three hundred ninety three
Bahdanau: seven hundred seventy seven million sixteen thousand three hundred ninety three

입력: 402018538
정답: four hundred two million eighteen thousand five hundred thirty eight
Luong: four hundred two million seventeen thou

## Greedy / Beam 비교

In [16]:
sample = test_loader.dataset.pairs[:200]

g = sum(
    greedy_decode(bahdanau_os_model, s, src_vocab, tgt_vocab) == t
    for s, t, _ in sample
)

b = sum(
    beam_decode(bahdanau_os_model, s, src_vocab, tgt_vocab, beam=5) == t
    for s, t, _ in sample
)

print(f"Greedy: {g/len(sample)*100:.2f}%")
print(f"Beam 5: {b/len(sample)*100:.2f}%")

Greedy: 100.00%
Beam 5: 99.50%


## 추가 실험: 1~2자리 숫자 오버샘플링

`ko2num` 브랜치에 있는 `oversample_short()`를 가져와 `max_digits=2`로 적용했습니다. 기존 Bahdanau 모델 구조와 학습·평가 함수는 변경하지 않았습니다.


In [10]:
OS_MAX_DIGITS = 2
OS_TARGET = 3000

def oversample_short(train_pairs, max_digits=OS_MAX_DIGITS, target=OS_TARGET):
    groups = defaultdict(list)
    for p in train_pairs:
        if len(p[2]) <= max_digits:
            groups[len(p[2])].append(p)
    extra = []
    for L, group in sorted(groups.items()):
        need = target - len(group)
        if need <= 0:
            continue
        reps, rem = divmod(need, len(group))
        extra += group * reps + random.Random(SEED + L).sample(group, rem)
    return extra

train_pairs = train_loader.dataset.pairs
extra_short = oversample_short(train_pairs)
train_loader_os, valid_loader_os, test_loader_os, src_vocab_os, tgt_vocab_os = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN, extra_train=extra_short
)

assert src_vocab_os.itos == src_vocab.itos
assert tgt_vocab_os.itos == tgt_vocab.itos
assert valid_loader_os.dataset.pairs == valid_loader.dataset.pairs
assert test_loader_os.dataset.pairs == test_loader.dataset.pairs
assert all(split_of(p[2]) == 'train' for p in extra_short)

for digits in [1, 2]:
    before = sum(len(p[2]) == digits for p in train_pairs)
    after = sum(len(p[2]) == digits for p in train_loader_os.dataset.pairs)
    print(f'{digits}자리: {before:,} -> {after:,}')
print(f'추가 학습 쌍: {len(extra_short):,}')
print(f'전체 Train: {len(train_loader.dataset):,} -> {len(train_loader_os.dataset):,}')
print(f'Valid: {len(valid_loader_os.dataset):,} / Test: {len(test_loader_os.dataset):,} (원본 유지)')


extra_train: 5,926개 추가 (valid/test와 겹쳐서 제외 0개)
1자리: 7 -> 3,000
2자리: 67 -> 3,000
추가 학습 쌍: 5,926
전체 Train: 80,028 -> 85,954
Valid: 9,967 / Test: 10,005 (원본 유지)


In [11]:
set_seed(SEED)

bahdanau_os_config = {**CONFIG, 'attention': 'bahdanau'}
bahdanau_os_model = Seq2Seq(
    len(src_vocab), len(tgt_vocab), **bahdanau_os_config
).to(DEVICE)

print('Attention:', bahdanau_os_model.decoder.attention)
print('Parameters:', sum(p.numel() for p in bahdanau_os_model.parameters()))

bahdanau_os_history = train(
    bahdanau_os_model,
    train_loader_os,
    valid_loader,
    epochs=EPOCHS,
    lr=LR
)


Attention: bahdanau
Parameters: 1857316
epoch  1 | train loss 0.4845 | valid loss 0.0142 | valid acc 96.30% | 55s
epoch  2 | train loss 0.0145 | valid loss 0.0018 | valid acc 99.74% | 50s
epoch  3 | train loss 0.0042 | valid loss 0.0009 | valid acc 99.85% | 50s
epoch  4 | train loss 0.0028 | valid loss 0.0001 | valid acc 99.98% | 50s
epoch  5 | train loss 0.0020 | valid loss 0.0001 | valid acc 99.99% | 50s
epoch  6 | train loss 0.0028 | valid loss 0.0010 | valid acc 99.79% | 50s
epoch  7 | train loss 0.0012 | valid loss 0.0001 | valid acc 99.99% | 49s
epoch  8 | train loss 0.0050 | valid loss 0.0001 | valid acc 99.96% | 50s
epoch  9 | train loss 0.0002 | valid loss 0.0000 | valid acc 99.99% | 50s
epoch 10 | train loss 0.0025 | valid loss 0.0001 | valid acc 99.97% | 49s
epoch 11 | train loss 0.0010 | valid loss 0.0001 | valid acc 99.98% | 50s
epoch 12 | train loss 0.0024 | valid loss 0.0002 | valid acc 99.98% | 49s
epoch 13 | train loss 0.0016 | valid loss 0.0000 | valid acc 99.99% | 49

In [12]:
bahdanau_os_rows = predict_all(bahdanau_os_model, test_loader, tgt_vocab)
bahdanau_os_acc = exact_match(bahdanau_os_rows)

print(f'기존 Bahdanau Test Accuracy (이전 실험): 99.98%')
print(f'1~2자리 Oversampling Test Accuracy: {bahdanau_os_acc*100:.2f}%')
print('Test Samples:', len(bahdanau_os_rows))
print('Best Valid Accuracy:', f"{max(h['valid_exact_match'] for h in bahdanau_os_history)*100:.2f}%")

print('\n[자릿수별 정확도]')
acc_by_length(bahdanau_os_rows)
print('\n[예측 사례]')
show_examples(bahdanau_os_rows, n=5)

print('\n[짧은 숫자 진단: Greedy / Beam 5]')
short_checks = {}
for s in [str(i) for i in range(10)]:
    pred = greedy_decode(bahdanau_os_model, s, src_vocab, tgt_vocab)
    short_checks[s] = pred
    print(f'{s} -> {pred}')
for s, expected in [('0', 'zero'), ('6', 'six')]:
    beam_pred = beam_decode(bahdanau_os_model, s, src_vocab, tgt_vocab, beam=5)
    print(f'{s}: greedy={short_checks[s]} | beam={beam_pred} | expected={expected}')


기존 Bahdanau Test Accuracy (이전 실험): 99.98%
1~2자리 Oversampling Test Accuracy: 99.97%
Test Samples: 10005
Best Valid Accuracy: 99.99%

[자릿수별 정확도]
자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |   1478 | 100.00%
     6 |   1522 | 100.00%
     7 |   1536 | 100.00%
     8 |   1472 | 100.00%
     9 |   1553 |  99.87%
    10 |   1448 |  99.93%

[예측 사례]
--- 맞은 예시 ---
입력: 897890
정답: eight hundred ninety seven thousand eight hundred ninety
예측: eight hundred ninety seven thousand eight hundred ninety

입력: 93532
정답: ninety three thousand five hundred thirty two
예측: ninety three thousand five hundred thirty two

입력: 3894895050
정답: three billion eight hundred ninety four million eight hundred ninety five thousand fifty
예측: three billion eight hundred ninety four million eight hundred ninety five thousand fifty

입력: 9471425031
정답: nine billion four hundred seventy one million four hundred twenty five thousand thirty one
예측

## Greedy / Beam 비교

In [ ]:
sample = test_loader.dataset.pairs[:200]

g = sum(
    greedy_decode(bahdanau_os_model, s, src_vocab, tgt_vocab) == t
    for s, t, _ in sample
)

b = sum(
    beam_decode(bahdanau_os_model, s, src_vocab, tgt_vocab, beam=5) == t
    for s, t, _ in sample
)

print(f"Greedy: {g/len(sample)*100:.2f}%")
print(f"Beam 5: {b/len(sample)*100:.2f}%")

## 최종 오버샘플링 모델 저장·재로드


In [15]:
import shutil

SAVE_PATH = os.path.join(SAVE_DIR, 'num2en.pt')
BACKUP_PATH = os.path.join(SAVE_DIR, 'num2en_before_oversampling.pt')
CANDIDATE_PATH = os.path.join(SAVE_DIR, 'num2en_oversampling_candidate.pt')

save_model(
    CANDIDATE_PATH,
    bahdanau_os_model,
    bahdanau_os_config,
    src_vocab,
    tgt_vocab,
    metrics={
        'test_exact_match': float(bahdanau_os_acc),
        'history': bahdanau_os_history,
        'oversampling': True,
        'oversample_max_digits': OS_MAX_DIGITS,
        'oversample_target_per_digit_length': OS_TARGET,
        'baseline_test_accuracy_recorded': 0.9998
    }
)

m, sv, tv, ckpt = load_model(CANDIDATE_PATH)
assert ckpt['config']['attention'] == 'bahdanau'
for s in ['0', '6', '12345', '8567012011']:
    assert greedy_decode(m, s, sv, tv) == greedy_decode(bahdanau_os_model, s, src_vocab, tgt_vocab)

print('Reload verified:', ckpt['config']['attention'])
print('Test Accuracy:', f"{ckpt['metrics']['test_exact_match']*100:.2f}%")
print('0:', greedy_decode(m, '0', sv, tv))
print('6:', greedy_decode(m, '6', sv, tv))

from google.colab import files

if short_checks['0'] == 'zero' and short_checks['6'] == 'six':
    if os.path.exists(SAVE_PATH) and not os.path.exists(BACKUP_PATH):
        shutil.copy2(SAVE_PATH, BACKUP_PATH)
        print('기존 모델 백업:', os.path.abspath(BACKUP_PATH))
    shutil.copy2(CANDIDATE_PATH, SAVE_PATH)
    print('Final model:', os.path.abspath(SAVE_PATH))
    files.download(SAVE_PATH)
else:
    print('짧은 숫자 번역 실패: 기존 num2en.pt를 덮어쓰지 않았습니다.')
    files.download(CANDIDATE_PATH)


saved: ../num2en_oversampling_candidate.pt (7.4 MB)
Reload verified: bahdanau
Test Accuracy: 99.97%
0: zero
6: six
Final model: /content/2026-nlp/project01/num2en.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>